# 🥉 Bronze — land the raw sales files

Reads **every CSV** in your bucket's `files/source/sales/` folder (upload them with **📁 My files**)
and writes them, **as they are**, to the table `my.bronze.sales`.

Bronze rule: **land, don't fix.** Every column stays text, exactly as in the file — the cleaning
happens in Silver. Two extra columns say where each row came from and when it was loaded.

In [ ]:
from pyspark.sql import functions as F

lake = spark.conf.get("spark.sql.catalog.my.warehouse")           # your lakehouse, e.g. ravi_lake
bucket = lake.removesuffix("_lake").replace("_", "-") + "-lake"   # your bucket,    e.g. ravi-lake
source = f"s3a://{bucket}/files/source/sales/"                    # every CSV in this folder
print("lakehouse:", lake, "· reading:", source)

## 1 · Read the files

In [ ]:
raw = (spark.read
       .option("header", True)      # first line = column names
       .csv(source))                # no schema → every column is text, nothing is lost

bronze = (raw
          .withColumn("_source_file", F.col("_metadata.file_name"))   # which file the row came from
          .withColumn("_ingested_at", F.current_timestamp()))        # when it was loaded

print(bronze.count(), "rows read")
bronze.show(5, truncate=False)

## 2 · Write the Bronze table
`createOrReplace()` = a full reload: the table always matches the files in the folder.

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS my.bronze")
bronze.writeTo("my.bronze.sales").using("iceberg").createOrReplace()
print("my.bronze.sales written")

## 3 · Check it — rows per file

In [ ]:
%%sql
SELECT _source_file, count(*) AS rows_loaded
FROM my.bronze.sales
GROUP BY _source_file
ORDER BY _source_file